# BGG GraphRAG QA

Semantic (vector) search over the board game knowledge graph — no SPARQL required.

**Pipeline:**
1. Load `bgg_main.ttl` + `fake_players.ttl` with rdflib
2. Convert each game / player to a natural-language text blurb
3. Embed with `sentence-transformers` (local, no API cost)
4. Store in two ChromaDB collections: `bgg_games` and `bgg_players`
5. At query time: embed the question, retrieve top-k blurbs, ask Claude

**First run:** expect ~2–3 min to build the index (21k enriched games).  
**Subsequent runs:** set `REBUILD_INDEX = False` in the config cell to skip embedding and go straight to queries.

In [ ]:
%pip install -q chromadb sentence-transformers tqdm

In [ ]:
import os
from pathlib import Path

from rdflib import Graph, Namespace, RDF, RDFS, Literal
from sentence_transformers import SentenceTransformer
import chromadb
from tqdm import tqdm
import anthropic

BGG   = Namespace("https://raw.githubusercontent.com/susvej/bg_ontology/")
TRENJ = Namespace("https://vejdemo.se/boardgames/threnjen#")

# ── Configuration ────────────────────────────────────────────────────────────
EMBED_MODEL        = "all-MiniLM-L6-v2"   # fast 384-dim; swap all-mpnet-base-v2 for quality
CHROMA_PATH        = "./chroma_bgg"
GAMES_COLLECTION   = "bgg_games"
PLAYERS_COLLECTION = "bgg_players"
ANSWER_MODEL       = "claude-haiku-4-5-20251001"
TOP_K_GAMES        = 8     # game results per query
TOP_K_PLAYERS      = 3     # player results per query
REBUILD_INDEX      = True  # set False after first run to skip embedding

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
print("Config OK")

## Build index
Run once. Set `REBUILD_INDEX = False` in the config cell to skip on re-runs.

In [ ]:
if REBUILD_INDEX:
    print("Loading bgg_main.ttl ...")
    g = Graph()
    g.parse("bgg_main.ttl", format="turtle")
    print(f"  {len(g):,} triples")

    print("Loading fake_players.ttl ...")
    g.parse("fake_players.ttl", format="turtle")
    print(f"  {len(g):,} triples total")
    print("Done loading.")
else:
    print("Skipping TTL load (REBUILD_INDEX=False).")

In [ ]:
def _lit(g, iri, pred):
    """Return string value of a datatype property, or None."""
    v = g.value(iri, pred)
    return str(v) if v is not None else None


def game_to_text(game_iri, g):
    name    = _lit(g, game_iri, BGG.hasName) or "Unknown"
    year    = _lit(g, game_iri, BGG.hasYearPublished)
    rating  = _lit(g, game_iri, BGG.hasRating)
    min_p   = _lit(g, game_iri, BGG.hasMinPlayers)
    max_p   = _lit(g, game_iri, BGG.hasMaxPlayers)
    best_p  = _lit(g, game_iri, BGG.hasBestNumPlayers)
    min_t   = _lit(g, game_iri, BGG.hasMinGameTime)
    max_t   = _lit(g, game_iri, BGG.hasMaxGameTime)
    age     = _lit(g, game_iri, BGG.hasMinRecAge)
    desc    = _lit(g, game_iri, BGG.hasDescription)
    enriched_val = g.value(game_iri, BGG.isFullyEnriched)
    enriched = bool(enriched_val) if enriched_val is not None else False

    header = name
    if year:
        header += f" ({year})"
    if rating:
        header += f" — Rating: {float(rating):.1f}/10"
    parts = [header]

    if min_p and max_p:
        pstr = f"Players: {min_p}–{max_p}"
        if best_p:
            pstr += f" (best: {best_p})"
        if min_t and max_t:
            pstr += f". Time: {min_t}–{max_t} min"
        elif min_t:
            pstr += f". Time: {min_t}+ min"
        if age:
            pstr += f". Age: {age}+"
        parts.append(pstr)

    if enriched:
        def labels(pred):
            return [str(g.value(v, RDFS.label) or "") for v in g.objects(game_iri, pred)
                    if g.value(v, RDFS.label)]
        cats    = labels(BGG.hasCategory)
        mechs   = labels(BGG.hasMechanic)
        themes  = labels(TRENJ.hasTheme)
        creators = labels(BGG.hasCreator)
        publishers = labels(BGG.hasPublisher)
        if cats:      parts.append("Categories: " + ", ".join(cats[:6]))
        if mechs:     parts.append("Mechanics: "  + ", ".join(mechs[:8]))
        if themes:    parts.append("Themes: "     + ", ".join(themes[:5]))
        if creators:  parts.append("Designers: "  + ", ".join(creators[:3]))
        if publishers: parts.append("Publisher: " + ", ".join(publishers[:2]))

    if desc:
        parts.append(desc[:400])

    return "\n".join(parts)


if REBUILD_INDEX:
    print("Generating game text blurbs ...")
    game_docs, game_ids, game_meta = [], [], []

    for game_iri in tqdm(list(g.subjects(RDF.type, BGG.Game))):
        gid = str(game_iri).rsplit("/", 1)[-1].rsplit("#", 1)[-1]
        text = game_to_text(game_iri, g)
        rating_val = g.value(game_iri, BGG.hasRating)
        enriched_val = g.value(game_iri, BGG.isFullyEnriched)
        game_docs.append(text)
        game_ids.append(f"game_{gid}")
        game_meta.append({
            "bgg_id":   gid,
            "name":     str(g.value(game_iri, BGG.hasName) or ""),
            "rating":   float(str(rating_val)) if rating_val else 0.0,
            "enriched": bool(enriched_val) if enriched_val is not None else False,
        })

    print(f"  {len(game_docs):,} game documents ready")

In [ ]:
if REBUILD_INDEX:
    print(f"Loading embedding model: {EMBED_MODEL}")
    model = SentenceTransformer(EMBED_MODEL)

    chroma = chromadb.PersistentClient(path=CHROMA_PATH)
    try:
        chroma.delete_collection(GAMES_COLLECTION)
        print("Cleared existing games collection")
    except Exception:
        pass
    games_col = chroma.create_collection(GAMES_COLLECTION)

    BATCH = 512
    print(f"Embedding {len(game_docs):,} games in batches of {BATCH} ...")
    for i in tqdm(range(0, len(game_docs), BATCH)):
        bd = game_docs[i:i+BATCH]
        bi = game_ids[i:i+BATCH]
        bm = game_meta[i:i+BATCH]
        be = model.encode(bd, show_progress_bar=False).tolist()
        games_col.add(documents=bd, embeddings=be, ids=bi, metadatas=bm)

    print(f"Games collection: {games_col.count():,} entries")
else:
    print("Loading embedding model (needed for queries) ...")
    model = SentenceTransformer(EMBED_MODEL)
    chroma = chromadb.PersistentClient(path=CHROMA_PATH)
    games_col = chroma.get_collection(GAMES_COLLECTION)
    print(f"Games collection loaded: {games_col.count():,} entries")

In [ ]:
def player_to_text(player_iri, g):
    name = str(g.value(player_iri, RDFS.label) or "Unknown Player")
    owned = list(g.objects(player_iri, BGG.hasOwnershipOf))
    owned_names = [str(g.value(gi, BGG.hasName) or gi) for gi in owned]

    opinions = []
    for op in g.subjects(BGG.hasOpinionHolder, player_iri):
        game_iri  = g.value(op, BGG.hasOpinionOf)
        gname     = str(g.value(game_iri, BGG.hasName) or "Unknown")
        rating    = g.value(op, BGG.hasPlayerRatingOpinion)
        load      = g.value(op, BGG.hasMentalLoad)
        load_str  = str(load).rsplit("/", 1)[-1] if load else None
        if rating:
            s = f"{gname} {int(float(str(rating)))}/10"
            if load_str:
                s += f" ({load_str})"
            opinions.append(s)

    parts = [f"{name} owns {len(owned)} games."]
    if owned_names:
        sample = owned_names[:8]
        tail   = f" and {len(owned_names)-8} more" if len(owned_names) > 8 else ""
        parts.append("Collection: " + ", ".join(sample) + tail)
    if opinions:
        parts.append("Ratings: " + "; ".join(opinions[:12]))
    return "\n".join(parts)


if REBUILD_INDEX:
    print("Generating player text blurbs ...")
    player_docs, player_ids, player_meta = [], [], []

    for p_iri in g.subjects(RDF.type, BGG.Player):
        pid  = str(p_iri).rsplit("/", 1)[-1].rsplit("#", 1)[-1]
        text = player_to_text(p_iri, g)
        player_docs.append(text)
        player_ids.append(f"player_{pid}")
        player_meta.append({"name": str(g.value(p_iri, RDFS.label) or "")})

    print(f"  {len(player_docs)} player documents ready")

    try:
        chroma.delete_collection(PLAYERS_COLLECTION)
    except Exception:
        pass
    players_col = chroma.create_collection(PLAYERS_COLLECTION)
    embs = model.encode(player_docs).tolist()
    players_col.add(documents=player_docs, embeddings=embs,
                    ids=player_ids, metadatas=player_meta)
    print(f"Players collection: {players_col.count()} entries")
else:
    players_col = chroma.get_collection(PLAYERS_COLLECTION)
    print(f"Players collection loaded: {players_col.count()} entries")

## Query
Run cells below any time — index must be built (or loaded) first.

In [ ]:
import textwrap

SYSTEM = """\
You are a helpful assistant answering questions about board games.
You are given context retrieved from a board game database: game descriptions
and player collection summaries.
Answer the question based on the provided context.
Be concise and direct. If the context does not contain enough information, say so.
Note: this is a semantic search system — results are ranked by similarity,
not sorted by rating or count, so avoid claiming exact rankings unless clearly
stated in the context.
"""

_PLAYER_KEYWORDS = {"player", "owns", "own", "collection", "who", "opinion", "rated", "rating"}


def ask(question: str,
        top_k_games: int = TOP_K_GAMES,
        top_k_players: int = TOP_K_PLAYERS) -> str:
    q_emb = model.encode([question])[0].tolist()

    # Always retrieve games
    g_res = games_col.query(query_embeddings=[q_emb], n_results=top_k_games)
    game_blurbs = g_res["documents"][0]

    # Retrieve players only if the question seems player-related
    q_lower = question.lower()
    include_players = any(kw in q_lower for kw in _PLAYER_KEYWORDS)
    player_blurbs = []
    if include_players:
        p_res = players_col.query(query_embeddings=[q_emb], n_results=top_k_players)
        player_blurbs = p_res["documents"][0]

    # Build context block
    ctx_parts = ["## Games\n"]
    for blurb in game_blurbs:
        ctx_parts.append(blurb)
        ctx_parts.append("---")
    if player_blurbs:
        ctx_parts.append("\n## Players\n")
        for blurb in player_blurbs:
            ctx_parts.append(blurb)
            ctx_parts.append("---")
    context = "\n".join(ctx_parts)

    response = client.messages.create(
        model=ANSWER_MODEL,
        max_tokens=512,
        system=SYSTEM,
        messages=[{"role": "user",
                   "content": f"Context:\n{context}\n\nQuestion: {question}"}],
    )
    answer = response.content[0].text.strip()
    print(f"Q: {question}\n")
    print(textwrap.fill(answer, width=80))
    print()
    return answer


print("ask() ready")

In [ ]:
ask("What are the top 10 highest-rated board games of all time?")

In [ ]:
ask("Which games use the Worker Placement mechanic and have a rating above 8?")

In [ ]:
ask("What games can be played by exactly 2 players and take less than 30 minutes?")

In [ ]:
ask("What games did Uwe Rosenberg design?")

In [ ]:
ask("What games did Uwe Rossenberg design?")
### Intentional misspelling of name

In [ ]:
ask("Which categories have the most games?")

In [ ]:
ask("What games are in the Fantasy category with a geek rating above 7.5?")

In [ ]:
ask("Which fake players gave a rating of 10 to any game, and what games did they rate?")

In [ ]:
# Change this and run
ask("What are the best cooperative games?")

In [ ]:
# Change this and run
ask("I like the game Set and the game Prosperitea. What other games with similar mechanics or categories might I like?")
### This is a great test to see if the agent will hallucinate when it doesn't have data. It does not!

In [ ]:
ask("I like the game Set and the game Ricochet Robots. What other games with similar mechanics or categories might I like?")
### this breaks! Fun! figure out why and fix it! (hint: check the mechanics and categories of these games in the data, and see how the query is structured to find similar games)

In [ ]:
ask("I want an 8 player game that is not a party game. What are the best options?")